# Stage 6: Final Forecast and Kaggle Submission

HistGradientBoosting was selected in Stage 5 with validation RMSLE **0.449788**, compared with Ridge at **0.494938** and the weekday baseline at **0.520631**. The final script reused the exact fixed specification, shifted the 365-day training window forward, built the shared matrices once, and fitted the model exactly once.

This notebook reads the generated artifacts without refitting or changing predictions.

In [1]:
from pathlib import Path
import pandas as pd

def find_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "submissions" / "store_sales_hgb_submission.csv").exists():
            return candidate
    raise FileNotFoundError("Run python -m src.final_forecast first")

ROOT = find_root()
submission_path = ROOT / "submissions" / "store_sales_hgb_submission.csv"
submission = pd.read_csv(submission_path)
checks = pd.read_csv(ROOT / "reports" / "tables" / "final_submission_checks.csv")
print("Final training: 2016-08-16 to 2017-08-15 (648,648 rows)")
print("Test: 2017-08-16 to 2017-08-31 (16 days, 28,512 rows)")
display(checks)
display(submission.head())
print(f"Manual Kaggle upload file: {submission_path}")

Final training: 2016-08-16 to 2017-08-15 (648,648 rows)
Test: 2017-08-16 to 2017-08-31 (16 days, 28,512 rows)


,check,value
0,test_row_count,28512
1,submission_row_count,28512
2,unique_test_ids,28512
3,unique_submission_ids,28512
4,ids_match,True
5,id_order_matches_test,True
6,id_order_matches_sample,True
7,missing_predictions,0
8,infinite_predictions,0
9,negative_predictions,0


,id,sales
0,3000888,3.607411
1,3000889,0.027124
2,3000890,4.726566
3,3000891,2000.024882
4,3000892,0.040672


Manual Kaggle upload file: C:\Users\GYDROC\Documents\Codex\2026-07-15\github-plugin-github-openai-curated-remote\work\stage1-repo\submissions\store_sales_hgb_submission.csv


## Leakage-safe final construction

Every lag lookup is forced to a date before `2017-08-16`; rolling summaries are fixed from known training history. Predicted test sales are never fed back into later features. The test-period integrity check passed, and predictions were restored to the original test and sample-submission ID order.

## Aggregate sanity checks

The plots are diagnostic only. Predictions were not manually rescaled or changed after inspection.

![Daily predicted sales](../reports/figures/final/01_test_daily_predictions.png)

![Highest predicted-sales families](../reports/figures/final/02_test_family_predictions.png)

## Limitations and next steps

The model was selected using one 16-day validation window, holiday inputs remain simplified and excluded, and intermittent families remain difficult. Kaggle leaderboard performance is not recorded. The next action is manual upload of `submissions/store_sales_hgb_submission.csv`; no automated submission is performed.